In [6]:
import nltk
import pandas as pd
import re
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from nltk.util import ngrams
from nltk.probability import FreqDist
from collections import defaultdict
import os

# --- 1. Configuration ---
# Let's take 50,000 documents. Adjust this if your machine can handle more/less.
NUM_DOCS_TO_TAKE = 9000000 
TRAIN_SIZE = 0.8  # 80% for training
VAL_SIZE = 0.1    # 10% for validation
TEST_SIZE = 0.1   # 10% for testing

# --- 2. Load and Stream a Subset ---
print(f"Streaming first {NUM_DOCS_TO_TAKE} documents from IndicCorpV2 (guj_Gujr)...")
dataset = load_dataset("ai4bharat/IndicCorpV2", "indiccorp_v2", streaming=True, split="guj_Gujr")
raw_texts = []

try:
    for i, doc in enumerate(dataset.take(NUM_DOCS_TO_TAKE)):
        raw_texts.append(doc['text'])
        if (i + 1) % 5000 == 0:
            print(f"  ...loaded {i+1} documents")
except Exception as e:
    print(f"Error during streaming (dataset might have changed or network issue): {e}")
    
print(f"Successfully loaded {len(raw_texts)} documents.")

# --- 3. Preprocessing and Tokenization ---
print("Cleaning and tokenizing documents...")

def clean_and_tokenize(text):
    """
    Simple cleaner and tokenizer for Gujarati.
    1. Removes non-Gujarati script and non-whitespace characters.
    2. Splits by whitespace.
    """
    # Keep only Gujarati script characters and whitespace
    text = re.sub(r'[^\u0A80-\u0AFF\s]', '', text)
    # Replace multiple whitespaces with a single space
    text = re.sub(r'\s+', ' ', text).strip()
    return text.split()

tokenized_sentences = [clean_and_tokenize(text) for text in raw_texts]
# Filter out any empty sentences that resulted from cleaning
tokenized_sentences = [s for s in tokenized_sentences if s]
# Create a parallel list of raw, cleaned sentences for TF-IDF
raw_sentences = [" ".join(s) for s in tokenized_sentences]

print(f"Processed into {len(raw_sentences)} sentences.")

# --- 4. Split Data ---
print("Splitting data into train, validation, and test sets...")

# First split: Train vs. (Val + Test)
train_raw, test_val_raw, train_tokens, test_val_tokens = train_test_split(
    raw_sentences, 
    tokenized_sentences, 
    test_size=(VAL_SIZE + TEST_SIZE), 
    random_state=42
)

# Second split: Val vs. Test
# (Original test_size) / (original test_size + val_size)
val_test_split_ratio = TEST_SIZE / (VAL_SIZE + TEST_SIZE)

val_raw, test_raw, val_tokens, test_tokens = train_test_split(
    test_val_raw, 
    test_val_tokens, 
    test_size=val_test_split_ratio, 
    random_state=42
)

print(f"Total sentences: {len(raw_sentences)}")
print(f"  Training sentences:   {len(train_raw)}")
print(f"  Validation sentences: {len(val_raw)}")
print(f"  Testing sentences:    {len(test_raw)}")

# --- 5. Generate N-gram Counts from TRAINING Data ---
print("\nGenerating unigram and bigram counts from TRAINING data...")

# Unigrams
all_train_tokens = [token for sent in train_tokens for token in sent]
unigram_counts = FreqDist(all_train_tokens)
print(f"  Found {len(unigram_counts)} unique unigrams.")

# Bigrams
all_train_bigrams = [bg for sent in train_tokens for bg in ngrams(sent, 2)]
bigram_counts = FreqDist(all_train_bigrams)
print(f"  Found {len(bigram_counts)} unique bigrams.")

# --- 6. Save N-gram Counts to CSV ---
print("Saving n-gram counts to CSV files...")

# Save Unigrams
unigram_df = pd.DataFrame(unigram_counts.items(), columns=['word', 'count'])
unigram_df = unigram_df.sort_values(by='count', ascending=False)
unigram_df.to_csv('unigram_counts.csv', index=False, encoding='utf-8-sig')
print("  Saved 'unigram_counts.csv'")

# Save Bigrams
# Convert tuple keys to separate columns
bigram_data = [
    {'word1': bg[0], 'word2': bg[1], 'count': count}
    for bg, count in bigram_counts.items()
]
bigram_df = pd.DataFrame(bigram_data)
bigram_df = bigram_df.sort_values(by='count', ascending=False)
bigram_df.to_csv('bigram_counts.csv', index=False, encoding='utf-8-sig')
print("  Saved 'bigram_counts.csv'")

# --- 7. Save Text Splits for Q2-Q4 ---
print("Saving raw text splits to .txt files for TF-IDF...")

def save_list_to_txt(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(item + '\n')

save_list_to_txt(train_raw, 'train_sentences.txt')
save_list_to_txt(val_raw, 'val_sentences.txt')
save_list_to_txt(test_raw, 'test_sentences.txt')
print("  Saved 'train_sentences.txt', 'val_sentences.txt', 'test_sentences.txt'")

print("\n--- Data Preparation Complete! ---")

Streaming first 9000000 documents from IndicCorpV2 (guj_Gujr)...
  ...loaded 5000 documents
  ...loaded 10000 documents
  ...loaded 15000 documents
  ...loaded 20000 documents
  ...loaded 25000 documents
  ...loaded 30000 documents
  ...loaded 35000 documents
  ...loaded 40000 documents
  ...loaded 45000 documents
  ...loaded 50000 documents
  ...loaded 55000 documents
  ...loaded 60000 documents
  ...loaded 65000 documents
  ...loaded 70000 documents
  ...loaded 75000 documents
  ...loaded 80000 documents
  ...loaded 85000 documents
  ...loaded 90000 documents
  ...loaded 95000 documents
  ...loaded 100000 documents
  ...loaded 105000 documents
  ...loaded 110000 documents
  ...loaded 115000 documents
  ...loaded 120000 documents
  ...loaded 125000 documents
  ...loaded 130000 documents
  ...loaded 135000 documents
  ...loaded 140000 documents
  ...loaded 145000 documents
  ...loaded 150000 documents
  ...loaded 155000 documents
  ...loaded 160000 documents
  ...loaded 165000 document

'_ssl.c:999: The handshake operation timed out' thrown while requesting GET https://huggingface.co/datasets/ai4bharat/IndicCorpV2/resolve/2d7285e6ce14fdb3fb2449c9f89427b9f582ac3f/data/gu.txt
Retrying in 1s [Retry 1/5].


  ...loaded 6890000 documents
  ...loaded 6895000 documents
  ...loaded 6900000 documents
  ...loaded 6905000 documents
  ...loaded 6910000 documents
  ...loaded 6915000 documents
  ...loaded 6920000 documents
  ...loaded 6925000 documents
  ...loaded 6930000 documents
  ...loaded 6935000 documents
  ...loaded 6940000 documents
  ...loaded 6945000 documents
  ...loaded 6950000 documents
  ...loaded 6955000 documents
  ...loaded 6960000 documents
  ...loaded 6965000 documents
  ...loaded 6970000 documents
  ...loaded 6975000 documents
  ...loaded 6980000 documents
  ...loaded 6985000 documents
  ...loaded 6990000 documents
  ...loaded 6995000 documents
  ...loaded 7000000 documents
  ...loaded 7005000 documents
  ...loaded 7010000 documents
  ...loaded 7015000 documents
  ...loaded 7020000 documents
  ...loaded 7025000 documents
  ...loaded 7030000 documents
  ...loaded 7035000 documents
  ...loaded 7040000 documents
  ...loaded 7045000 documents
  ...loaded 7050000 documents
  ...loade

In [1]:
import nltk
import math
import numpy as np
import pandas as pd
from nltk.util import ngrams
from nltk.probability import FreqDist
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import time

# === Helper function to load data from files ===

def load_text_file(filename):
    """Loads a .txt file, returning a list of strings."""
    with open(filename, 'r', encoding='utf-8') as f:
        # Read lines and strip newline characters
        return [line.strip() for line in f]

def load_tokenized_data(raw_texts):
    """Tokenizes a list of raw sentences."""
    # We re-use the same simple tokenizer
    return [text.split() for text in raw_texts]

def load_counts_from_csv(unigram_file, bigram_file):
    """Loads n-gram counts from CSVs into FreqDist-like dicts."""
    print("Loading n-gram counts from CSV...")
    # Load Unigrams
    unigram_df = pd.read_csv(unigram_file)
    # Convert to a dictionary: {'word': count}
    unigram_counts = dict(zip(unigram_df['word'], unigram_df['count']))
    
    # Load Bigrams
    bigram_df = pd.read_csv(bigram_file)
    # Convert to a dictionary: {('word1', 'word2'): count}
    bigram_counts = dict(zip(
        zip(bigram_df['word1'], bigram_df['word2']), 
        bigram_df['count']
    ))
    
    print(f"Loaded {len(unigram_counts)} unigrams and {len(bigram_counts)} bigrams.")
    return defaultdict(int, unigram_counts), defaultdict(int, bigram_counts)

# --- Prerequisite: Load Data from Files ---
print("Loading data from generated files...")

# For Q2, 3, 4 (TF-IDF)
train_raw = load_text_file('train_sentences.txt')
val_raw = load_text_file('val_sentences.txt')
test_raw = load_text_file('test_sentences.txt')

# For Q1 (PMI)
train_tokenized = load_tokenized_data(train_raw)
val_tokenized = load_tokenized_data(val_raw)
test_tokenized = load_tokenized_data(test_raw)

print(f"Loaded {len(train_raw)} train, {len(val_raw)} val, {len(test_raw)} sentences.\n")

# ======================================================================
# Question 1: Compute PMI Scores
# ======================================================================
print("--- Starting Question 1: PMI Scores ---")

# 1.1: Load unigram and bigram counts from CSV
unigram_counts, bigram_counts = load_counts_from_csv(
    'unigram_counts.csv', 
    'bigram_counts.csv'
)

# We need the *total* counts, which we can get from the dicts
N_tokens = sum(unigram_counts.values())
N_bigrams = sum(bigram_counts.values())

print(f"Total tokens in train: {N_tokens}")
print(f"Total bigrams in train: {N_bigrams}")

# 1.2: Define PMI calculation function
def compute_ppmi(w1, w2, unigram_counts, bigram_counts, N_tokens, N_bigrams):
    count_w1 = unigram_counts[w1]
    count_w2 = unigram_counts[w2]
    count_w1_w2 = bigram_counts[(w1, w2)]
    
    if count_w1 == 0 or count_w2 == 0 or count_w1_w2 == 0:
        return 0.0

    # Note: Using N_bigrams for P(w1, w2) might be slightly less accurate
    # than N_tokens, but let's stick to the counts we have.
    # A more standard PMI uses P(w1,w2) = count(w1,w2) / N_tokens
    # Let's adjust to use N_tokens as the denominator for all probs
    
    prob_w1 = count_w1 / N_tokens
    prob_w2 = count_w2 / N_tokens
    prob_w1_w2 = count_w1_w2 / N_tokens # P(w1, w2)

    if prob_w1 == 0 or prob_w2 == 0 or prob_w1_w2 == 0:
        return 0.0

    pmi = math.log2(prob_w1_w2 / (prob_w1 * prob_w2))
    return max(0, pmi)

# 1.3: Compute PMI for all bigrams in validation and test sets
val_bigrams_set = set(bg for sent in val_tokenized for bg in ngrams(sent, 2))
test_bigrams_set = set(bg for sent in test_tokenized for bg in ngrams(sent, 2))

print(f"\nCalculating PMI for {len(val_bigrams_set)} Validation Set Bigrams:")
val_pmi_scores = {}
for bg in val_bigrams_set:
    pmi = compute_ppmi(bg[0], bg[1], unigram_counts, bigram_counts, N_tokens, N_bigrams)
    val_pmi_scores[bg] = pmi
# (Printing all is too much, we'll just show a few)
print("  Top 5 PMI scores in Validation set (sample):")
for bg, pmi in sorted(val_pmi_scores.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f"    PPMI({bg[0]}, {bg[1]}): {pmi:.4f}")


print(f"\nCalculating PMI for {len(test_bigrams_set)} Test Set Bigrams:")
test_pmi_scores = {}
for bg in test_bigrams_set:
    pmi = compute_ppmi(bg[0], bg[1], unigram_counts, bigram_counts, N_tokens, N_bigrams)
    test_pmi_scores[bg] = pmi
print("  Top 5 PMI scores in Test set (sample):")
for bg, pmi in sorted(test_pmi_scores.items(), key=lambda item: item[1], reverse=True)[:5]:
    print(f"    PPMI({bg[0]}, {bg[1]}): {pmi:.4f}")


# ======================================================================
# Question 2: Vectorize using TF-IDF
# ======================================================================
print("\n--- Starting Question 2: TF-IDF Vectorization ---")

# We use the raw_sentences lists loaded from the .txt files
tfidf_vectorizer = TfidfVectorizer()

print("Fitting TF-IDF on training data...")
tfidf_vectorizer.fit(train_raw)
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

X_train_tfidf = tfidf_vectorizer.transform(train_raw)
X_val_tfidf = tfidf_vectorizer.transform(val_raw)
X_test_tfidf = tfidf_vectorizer.transform(test_raw)

print(f"Shape of train TF-IDF matrix: {X_train_tfidf.shape}")
print(f"Shape of val TF-IDF matrix: {X_val_tfidf.shape}")
print(f"Shape of test TF-IDF matrix: {X_test_tfidf.shape}")

# ======================================================================
# Question 3: Nearest Neighbor (Intra-set)
# ======================================================================
print("\n--- Starting Question 3: Intra-Set Nearest Neighbors ---")

def find_intra_set_neighbors(tfidf_matrix, raw_texts, set_name=""):
    print(f"\nCalculating neighbors for {set_name} set...")
    if tfidf_matrix.shape[0] < 2:
        print("Not enough sentences in the set to find a neighbor.")
        return

    sim_matrix = cosine_similarity(tfidf_matrix)
    np.fill_diagonal(sim_matrix, -1.0)
    
    nearest_neighbor_indices = np.argmax(sim_matrix, axis=1)
    
    print(f"  Showing results for first 3 sentences in {set_name}:")
    for i in range(min(3, len(raw_texts))): # Show first 3
        nn_idx = nearest_neighbor_indices[i]
        similarity = sim_matrix[i, nn_idx]
        
        print(f"\n    Query ({set_name} {i}): '{raw_texts[i][:80]}...'")
        print(f"    Neighbor ({set_name} {nn_idx}): '{raw_texts[nn_idx][:80]}...'")
        print(f"    Similarity: {similarity:.4f}")

find_intra_set_neighbors(X_val_tfidf, val_raw, "Validation")
find_intra_set_neighbors(X_test_tfidf, test_raw, "Test")

# ======================================================================
# Question 4: Nearest Neighbor (Inter-set) & Bonus
# ======================================================================
print("\n--- Starting Question 4: Inter-Set Nearest Neighbors (Bonus) ---")
print("Using sklearn.neighbors.NearestNeighbors for efficient search.")

nn_model = NearestNeighbors(n_neighbors=1, algorithm='brute', metric='cosine')

print("Fitting NearestNeighbors model on training data...")
nn_model.fit(X_train_tfidf)

# --- Find neighbors for Validation set (from Training set) ---
print("\nSearching for Val neighbors in Train data...")
val_distances, val_neighbor_indices = nn_model.kneighbors(X_val_tfidf)

print("  Showing results for first 3 sentences in Validation set:")
for i in range(min(3, len(val_raw))):
    nn_idx = val_neighbor_indices[i][0]
    dist = val_distances[i][0]
    
    print(f"\n    Query (Val {i}): '{val_raw[i][:80]}...'")
    print(f"    Neighbor (Train {nn_idx}): '{train_raw[nn_idx][:80]}...'")
    print(f"    Cosine Distance: {dist:.4f} (Similarity = {1 - dist:.4f})")

# --- Find neighbors for Test set (from Training set) ---
print("\nSearching for Test neighbors in Train data...")
test_distances, test_neighbor_indices = nn_model.kneighbors(X_test_tfidf)

print("  Showing results for first 3 sentences in Test set:")
for i in range(min(3, len(test_raw))):
    nn_idx = test_neighbor_indices[i][0]
    dist = test_distances[i][0]
    
    print(f"\n    Query (Test {i}): '{test_raw[i][:80]}...'")
    print(f"    Neighbor (Train {nn_idx}): '{train_raw[nn_idx][:80]}...'")
    print(f"    Cosine Distance: {dist:.4f} (Similarity = {1 - dist:.4f})")

print("\n--- All tasks complete. ---")

Loading data from generated files...
Loaded 20000 train, 2500 val, 2500 sentences.

--- Starting Question 1: PMI Scores ---
Loading n-gram counts from CSV...
Loaded 96866 unigrams and 501213 bigrams.
Total tokens in train: 813515
Total bigrams in train: 793515

Calculating PMI for 77133 Validation Set Bigrams:
  Top 5 PMI scores in Validation set (sample):
    PPMI(તા૯૧૧૯ના, ઠરાવથી): 19.6338
    PPMI(હરિત, ક્રાંતિને): 19.6338
    PPMI(ઘીનું, મોણ): 19.6338
    PPMI(બ્રિટની, સ્પીયર્સ): 19.6338
    PPMI(સંજૂ, સેમસનને): 19.6338

Calculating PMI for 79913 Test Set Bigrams:
  Top 5 PMI scores in Test set (sample):
    PPMI(હિતેન, તેજવાની): 19.6338
    PPMI(ગૂજરાત, વિદ્યાપીઠનાં): 19.6338
    PPMI(બેન્જામિન, મૂરે): 19.6338
    PPMI(કન્ઝયુમર, ડયુરેબલ્સ): 19.6338
    PPMI(વઘારવાનું, તપેલું): 19.6338

--- Starting Question 2: TF-IDF Vectorization ---
Fitting TF-IDF on training data...
Vocabulary size: 11511
Shape of train TF-IDF matrix: (20000, 11511)
Shape of val TF-IDF matrix: (2500, 11511)
Sha